# SecureLog AI — Anomaly Detection Experiments

Exploratory notebook used during development to try out feature choices and
model settings before they were locked into `app/features.py` and
`app/model.py`. This is the "lab notebook" companion to the production
pipeline — it's meant for experimentation, not for reuse in the app.

**Contents**
1. Load and inspect the sample log data
2. Parse raw logs with `app.parser`
3. Explore feature engineering choices
4. Compare Isolation Forest against Local Outlier Factor
5. Visualize anomaly scores


In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from app import parser, features, model

pd.set_option("display.max_colwidth", 80)


## 1. Load the sample log data

In [ ]:
raw_path = "../data/sample_logs.csv"
with open(raw_path, "rb") as f:
    raw_bytes = f.read()

parsed = parser.parse_csv_log(raw_bytes)
print(f"Parsed {parsed.parsed_lines}/{parsed.total_lines} lines (rate={parsed.parse_rate:.1%})")
parsed.df.head()


## 2. Quick look at the class balance we're working with

Logs are naturally imbalanced — most events are routine. That's exactly why an unsupervised approach (Isolation Forest) was chosen over a supervised classifier, which would need labeled anomalies we don't have.

In [ ]:
parsed.df["level"].value_counts()


## 3. Feature engineering

Build the same feature matrix the production app uses, so this notebook stays a faithful test bed rather than a diverging copy.

In [ ]:
feature_df, scaler, vectorizer, svd = features.build_feature_matrix(parsed.df)
print(feature_df.shape)
feature_df.head()


## 4. Try Isolation Forest at a few contamination levels

`contamination` is the model's prior on what fraction of events are anomalous. We don't know the true rate for a new log source, so it's worth sanity-checking a couple of values before picking a default.

In [ ]:
for c in [0.02, 0.05, 0.1]:
    scores = model.run_detection(feature_df, contamination=c)
    print(f"contamination={c:>4} -> flagged {scores['is_anomaly'].sum():>3} / {len(scores)} events")


`contamination=0.05` was chosen as the app default — it flagged the injected brute-force burst and the scattered anomalies in the synthetic dataset without over-flagging routine traffic. Real deployments should tune this per log source (see `docs/architecture.md`).

In [ ]:
scores = model.run_detection(feature_df, contamination=0.05)
results = pd.concat([parsed.df.reset_index(drop=True), scores.reset_index(drop=True)], axis=1)
results.sort_values("anomaly_score", ascending=False).head(10)[["timestamp", "level", "source", "message", "anomaly_score"]]


## 5. Compare against Local Outlier Factor

LOF was considered as an alternative. It's density-based rather than tree-based, which can help with local clusters of anomalies, but it doesn't naturally support scoring new/unseen events without refitting — a real limitation for a system meant to score a continuous log stream. Isolation Forest was kept as the production default; this comparison is here for the record.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_labels = lof.fit_predict(feature_df)
print(f"LOF flagged {(lof_labels == -1).sum()} / {len(lof_labels)} events as anomalous")

agreement = (lof_labels == -1) & results["is_anomaly"].values
print(f"Agreement with Isolation Forest: {agreement.sum()} events flagged by both")


## 6. Visualize anomaly scores over the event sequence

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
normal = results[~results["is_anomaly"]]
anomalous = results[results["is_anomaly"]]

ax.scatter(normal.index, normal["anomaly_score"], s=10, alpha=0.5, label="Normal")
ax.scatter(anomalous.index, anomalous["anomaly_score"], s=30, alpha=0.9, label="Anomaly", color="crimson")
ax.set_xlabel("Event index")
ax.set_ylabel("Anomaly score (0-100)")
ax.set_title("Isolation Forest anomaly scores — sample_logs.csv")
ax.legend()
plt.tight_layout()
plt.show()


## Notes / next experiments

- Try weighting the text-similarity features more heavily once we have a larger, noisier real-world log sample.
- Evaluate a rolling/online variant so the model doesn't need to be refit from scratch on every batch.
- Add a small labeled validation set (even 50-100 hand-labeled events) to measure precision/recall instead of only eyeballing the flagged list.